In [1]:
import pandas as pd
from sqlalchemy import create_engine
from DATA.stock_invest_function import *


In [2]:
def add_yoy_growth(df, value_column='value', group_column='root_hs_code', date_column='date'):
    """
    root_hs_code별로 value 컬럼의 연간 증가율을 계산하여 새로운 컬럼으로 추가합니다.
    """
    df = df.copy()
    df[date_column] = pd.to_datetime(df[date_column])
    df.sort_values(by=[group_column, date_column], inplace=True)

    # YoY (12개월 전 대비 비율 변화율) 계산
    df[f'{value_column}_yoy'] = (
        df.groupby(group_column)[value_column]
        .transform(lambda x: x.pct_change(periods=12))
    )

    return df

In [4]:
db_info = {
    'user': 'stox7412',         # 예: 'root'
    'password': 'Apt106503!~', # 예: '1234'
    # 'host' : '192.168.0.230',
    'host': get_db_host(),         # 예: 'localhost' 또는 IP
    'port': '3307',              # 기본 포트는 보통 3306
    'database': 'investar'        # 예: 'trade_data'
}

trade_df = fetch_table_data(db_info, "korea_monthly_trade_act_forecast_data")

df_combined_with_yoy = add_yoy_growth(trade_df)

company_num = 490

# 1. 예측 데이터만 필터링하고 NaN 제거
df_forecast_only = df_combined_with_yoy[
    (df_combined_with_yoy['is_forecast'] == 1) &
    (df_combined_with_yoy['value_yoy'].notna())
]

# 2. root_hs_code별 평균 YoY 증가율 계산
avg_yoy_by_code = (
    df_forecast_only
    .groupby('root_hs_code')['value_yoy']
    .mean()
    .reset_index()
    .rename(columns={'value_yoy': 'avg_forecast_yoy'})
)

# 3. 수출 증가율 기준으로 상위 10개 추출
top_company_codes = avg_yoy_by_code.sort_values(by='avg_forecast_yoy', ascending=False).head(company_num)

✅ 'korea_monthly_trade_act_forecast_data' 테이블에서 121600건의 데이터를 가져왔습니다.


In [5]:
top_company_codes[top_company_codes['root_hs_code'] == '330590']

,root_hs_code,avg_forecast_yoy
68,330590,0.095875


In [6]:
## 강소기업 데이터 가져오기

In [7]:
company_df = fetch_table_data(db_info, 'hs_code_by_kr_monster_company')
company_df.rename(columns={'hs_code_6d': 'root_hs_code'}, inplace=True)

✅ 'hs_code_by_kr_monster_company' 테이블에서 185건의 데이터를 가져왔습니다.


In [9]:
monster_df = pd.merge(company_df, top_company_codes, on='root_hs_code', how='left', indicator=True)

In [11]:
monster_df[monster_df['Name'] == '삼성전기']

,hs_code,품목명,Code,Name,root_hs_code,avg_forecast_yoy,_merge
141,8532240000,적층세라믹콘덴서,A009150,삼성전기,853224,0.035591,both


In [ ]:
hscode = fetch_table_data(db_info, "korea_fs_data")